In [1]:
# Retail Demand Forecasting & Inventory Optimization
# Notebook 02: Feature Engineering

print("Retail Demand Forecasting & Inventory Optimization")
print("Feature Engineering")

Retail Demand Forecasting & Inventory Optimization
Feature Engineering


In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)

In [3]:
PROJECT_ROOT = Path("..")

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "clean_train.csv"
)

df = pd.read_csv(DATA_PATH)

df["date"] = pd.to_datetime(df["date"])

print("Cleaned dataset loaded successfully.")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

Cleaned dataset loaded successfully.
Rows: 248,690
Columns: 8


In [4]:
df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,demand,date
0,FOODS_1_021_CA_1_evaluation,FOODS_1_021,FOODS_1,FOODS,CA_1,CA,0,2011-01-02
1,FOODS_1_021_CA_1_evaluation,FOODS_1_021,FOODS_1,FOODS,CA_1,CA,0,2011-01-03
2,FOODS_1_021_CA_1_evaluation,FOODS_1_021,FOODS_1,FOODS,CA_1,CA,0,2011-01-04
3,FOODS_1_021_CA_1_evaluation,FOODS_1_021,FOODS_1,FOODS,CA_1,CA,0,2011-01-05
4,FOODS_1_021_CA_1_evaluation,FOODS_1_021,FOODS_1,FOODS,CA_1,CA,0,2011-01-06


In [5]:
df = df.sort_values(
    ["item_id", "store_id", "date"]
).reset_index(drop=True)

print("Data sorted by product, store, and date.")

Data sorted by product, store, and date.


In [6]:
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_month"] = df["date"].dt.day
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter
df["year"] = df["date"].dt.year
df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype(int)

print("Calendar features created.")

Calendar features created.


In [7]:
calendar_features = [
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "quarter",
    "year",
    "is_weekend"
]

df[calendar_features].head(10)

,day_of_week,day_of_month,week_of_year,month,quarter,year,is_weekend
0,6,2,52,1,1,2011,1
1,0,3,1,1,1,2011,0
2,1,4,1,1,1,2011,0
3,2,5,1,1,1,2011,0
4,3,6,1,1,1,2011,0
5,4,7,1,1,1,2011,0
6,5,8,1,1,1,2011,1
7,6,9,1,1,1,2011,1
8,0,10,2,1,1,2011,0
9,1,11,2,1,1,2011,0


In [8]:
group_columns = ["item_id", "store_id"]

df["lag_1"] = (
    df.groupby(group_columns)["demand"]
    .shift(1)
)

df["lag_7"] = (
    df.groupby(group_columns)["demand"]
    .shift(7)
)

df["lag_14"] = (
    df.groupby(group_columns)["demand"]
    .shift(14)
)

df["lag_28"] = (
    df.groupby(group_columns)["demand"]
    .shift(28)
)

print("Lag features created:")
print("lag_1, lag_7, lag_14, lag_28")

Lag features created:
lag_1, lag_7, lag_14, lag_28


In [9]:
df["rolling_mean_7"] = (
    df.groupby(group_columns)["demand"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=7,
            min_periods=1
        ).mean()
    )
)

df["rolling_mean_28"] = (
    df.groupby(group_columns)["demand"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=28,
            min_periods=1
        ).mean()
    )
)

print("Rolling mean features created.")

Rolling mean features created.


In [10]:
df["rolling_std_7"] = (
    df.groupby(group_columns)["demand"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=7,
            min_periods=1
        ).std()
    )
)

df["rolling_std_28"] = (
    df.groupby(group_columns)["demand"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=28,
            min_periods=1
        ).std()
    )
)

print("Rolling standard deviation features created.")

Rolling standard deviation features created.


In [11]:
feature_columns = [
    "day_of_week",
    "day_of_month",
    "week_of_year",
    "month",
    "quarter",
    "year",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28"
]

print("Feature columns:")
for feature in feature_columns:
    print("-", feature)

Feature columns:
- day_of_week
- day_of_month
- week_of_year
- month
- quarter
- year
- is_weekend
- lag_1
- lag_7
- lag_14
- lag_28
- rolling_mean_7
- rolling_mean_28
- rolling_std_7
- rolling_std_28


In [12]:
feature_summary = df[
    feature_columns
].describe().T

display(feature_summary)

,count,mean,std,min,25%,50%,75%,max
day_of_week,248690.0,3.000000,2.001310,0.0,1.0,3.000000,5.000000,6.000000
day_of_month,248690.0,15.710925,8.782787,1.0,8.0,16.000000,23.000000,31.000000
week_of_year,248690.0,25.725039,15.322925,1.0,12.0,25.000000,39.000000,53.000000
month,248690.0,6.316780,3.504170,1.0,3.0,6.000000,9.000000,12.000000
quarter,248690.0,2.440146,1.135698,1.0,1.0,2.000000,3.000000,4.000000
year,248690.0,2013.138526,1.516990,2011.0,2012.0,2013.000000,2014.000000,2016.000000
is_weekend,248690.0,0.285938,0.451861,0.0,0.0,0.000000,1.000000,1.000000
lag_1,248560.0,0.755355,1.933115,0.0,0.0,0.000000,1.000000,183.000000
lag_7,247780.0,0.754552,1.932729,0.0,0.0,0.000000,1.000000,183.000000
lag_14,246870.0,0.753599,1.933265,0.0,0.0,0.000000,1.000000,183.000000


In [13]:
lag_columns = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28"
]

missing_features = df[lag_columns].isnull().sum()

display(
    missing_features.to_frame(
        name="missing_values"
    )
)

,missing_values
lag_1,130
lag_7,910
lag_14,1820
lag_28,3640
rolling_mean_7,130
rolling_mean_28,130
rolling_std_7,260
rolling_std_28,260


In [15]:
before_rows = len(df)

df_features = df.dropna(
    subset=lag_columns
).copy()

after_rows = len(df_features)

print(f"Rows before feature cleaning: {before_rows:,}")
print(f"Rows after feature cleaning : {after_rows:,}")
print(f"Rows removed                : {before_rows - after_rows:,}")

Rows before feature cleaning: 248,690
Rows after feature cleaning : 245,050
Rows removed                : 3,640


In [16]:
df_features.head()

,id,item_id,dept_id,cat_id,store_id,state_id,demand,date,day_of_week,day_of_month,week_of_year,month,quarter,year,is_weekend,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28,rolling_std_7,rolling_std_28
28,FOODS_1_021_CA_1_evaluation,FOODS_1_021,FOODS_1,FOODS,CA_1,CA,0,2011-01-30,6,30,4,1,1,2011,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
29,FOODS_1_021_CA_1_evaluation,FOODS_1_021,FOODS_1,FOODS,CA_1,CA,0,2011-01-31,0,31,5,1,1,2011,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30,FOODS_1_021_CA_1_evaluation,FOODS_1_021,FOODS_1,FOODS,CA_1,CA,0,2011-02-01,1,1,5,2,1,2011,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
31,FOODS_1_021_CA_1_evaluation,FOODS_1_021,FOODS_1,FOODS,CA_1,CA,0,2011-02-02,2,2,5,2,1,2011,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
32,FOODS_1_021_CA_1_evaluation,FOODS_1_021,FOODS_1,FOODS,CA_1,CA,0,2011-02-03,3,3,5,2,1,2011,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
print("Final feature-engineered dataset")
print(f"Rows    : {len(df_features):,}")
print(f"Columns : {len(df_features.columns)}")

Final feature-engineered dataset
Rows    : 245,050
Columns : 23


In [18]:
correlation = (
    df_features[
        feature_columns + ["demand"]
    ]
    .corr()["demand"]
    .sort_values(
        ascending=False
    )
)

display(correlation)

demand             1.000000
rolling_mean_28    0.591508
rolling_mean_7     0.559470
rolling_std_28     0.514923
rolling_std_7      0.459344
lag_1              0.396994
lag_7              0.393722
lag_14             0.389305
lag_28             0.384166
year               0.082936
is_weekend         0.038453
day_of_week        0.015400
week_of_year       0.001127
month              0.000332
quarter           -0.001572
day_of_month      -0.003105
Name: demand, dtype: float64

In [19]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_model_features.csv"
)

df_features.to_csv(
    OUTPUT_PATH,
    index=False
)

print("Feature-engineered dataset saved:")
print(OUTPUT_PATH)

Feature-engineered dataset saved:
..\data\processed\notebook_model_features.csv


In [20]:
print("""
FEATURE ENGINEERING SUMMARY
----------------------------

Calendar features:
- Day of week
- Day of month
- Week of year
- Month
- Quarter
- Year
- Weekend indicator

Lag features:
- 1-day lag
- 7-day lag
- 14-day lag
- 28-day lag

Rolling features:
- 7-day rolling mean
- 28-day rolling mean
- 7-day rolling standard deviation
- 28-day rolling standard deviation

These features provide historical demand and time-based
information required by the forecasting models.
""")


FEATURE ENGINEERING SUMMARY
----------------------------

Calendar features:
- Day of week
- Day of month
- Week of year
- Month
- Quarter
- Year
- Weekend indicator

Lag features:
- 1-day lag
- 7-day lag
- 14-day lag
- 28-day lag

Rolling features:
- 7-day rolling mean
- 28-day rolling mean
- 7-day rolling standard deviation
- 28-day rolling standard deviation

These features provide historical demand and time-based
information required by the forecasting models.

